# Tierra — ¿qué aeronave puede aterrizar aquí?
**Inspira STEM 2026 · Aeronáutica para el futuro**
**Equipo 2**

---

La Tierra es el caso de referencia: atmósfera densa, medida con precisión y con sesenta años de experiencia en frenarla. Detenerse no es el problema. El problema es el calor.

Regresar de órbita significa disipar la energía de 7 800 m/s, y casi toda termina como calor sobre el escudo en menos de dos minutos. Por eso el corredor de reentrada es tan estrecho: unos pocos grados separan quemarse de rebotar hacia el espacio.

Tu equipo no decide si alguna aeronave puede aterrizar. Decide cuál lo hace sin salirse de los límites del escudo.

---

## Cómo se trabaja

Tienes cinco aeronaves y un solo planeta. El archivo evalúa **una aeronave a la vez**.

1. Ejecuta el Paso 1 una sola vez.
2. Escribe en el Paso 2 los datos de la primera aeronave.
3. Ejecuta las tres fases en orden. Cada fase le entrega a la siguiente la velocidad y la altura con que termina, igual que en una misión real.
4. Copia el resumen final a tu ficha de decisión.
5. Vuelve al Paso 2, cambia de aeronave y repite.

Cinco pasadas, cinco filas, una decisión que defender.

## Cómo se decide

El archivo evalúa ocho límites. La regla es la que usa la industria:

> **Gana la aeronave que cumple los ocho límites. Si más de una los cumple, gana la más ligera.**

La masa es la moneda de todo vuelo espacial: cada kilogramo que se lanza cuesta combustible en cada etapa anterior de la misión. Una nave sobredimensionada que cumple todo no es una buena solución, es una solución cara.

Puede que en tu planeta solo una aeronave cumpla los ocho. Puede que cumplan varias y tengas que elegir por masa. Y puede que no cumpla ninguna, en cuyo caso tu trabajo es señalar exactamente qué lo impide.

## Paso 1 — El entorno

No modifiques nada de esta celda. Es lo que te tocó.

| | |
|---|---|
| Gravedad | 9,81 m/s² |
| Densidad en superficie | 1,225 kg/m³ |
| Escala de altura | 8,500 m |
| Velocidad del sonido | 340 m/s |
| Velocidad de entrada | 7,800 m/s |
| Interfaz de entrada | 120 km |
| Ángulo de entrada | 2,5° |

Gravedad, densidad y escala de altura son los valores estándar a nivel del mar. La velocidad del sonido, 340 m/s, corresponde a aire seco a 15 °C. La velocidad de entrada es la de una cápsula que regresa de órbita baja, y la interfaz de entrada de 120 km es la altitud convencional a la que se considera que empieza la reentrada.

**Sobre el ángulo de entrada.** No es una variable de diseño: lo fija la navegación de la misión mucho antes de tocar la atmósfera, y es igual para las cinco aeronaves. Apolo y Soyuz reentran con ángulos de entre 2° y 7°. Un ángulo mayor acorta el vuelo pero multiplica la carga: a 9° ninguna aeronave de la flota sobrevive.

La atmósfera se modela como exponencial: la densidad se reduce a la mitad cada vez que se sube una escala de altura.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':110,'font.size':9,'axes.grid':True,'grid.alpha':.3,'lines.linewidth':2})

PLANETA, G, RHO0, ESCALA, SONIDO = "Tierra", 9.81, 1.225, 8500.0, 340.0
V_ENTRADA, H_ENTRADA, GAMMA = 7800.0, 120000.0, 2.5
MACH_DOSEL, H_SUELTA, HOVER = 2., 1800., 30.

def volar(h0, v0, hf, m, cd, area, gam=90., suelta=0., t_suelta=1e9, mach=None):
    r = np.radians(gam); t, h, v, mm = 0., h0, v0, m
    T = {k: [] for k in ["t","h","v","mach","q","f","n","vt"]}
    while h > hf and v > 0. and t < 3e4:
        if mach and v/SONIDO <= mach: break
        if t > t_suelta: mm = m - suelta
        rho = RHO0*np.exp(-h/ESCALA); q = .5*rho*v*v; d = q*cd*area; a = G*np.sin(r) - d/mm
        for k, x in zip(T, [t, h, v, v/SONIDO, q, d/1000, d/(mm*9.81),
                            np.sqrt(2*mm*G*np.sin(r)/max(rho*cd*area, 1e-12))]): T[k].append(x)
        dt = min(5., max(.002, 1./(.2+abs(a)))); v += a*dt; h -= v*np.sin(r)*dt; t += dt
    return {k: np.array(x) for k, x in T.items()}, v, h, t

print(f"Entorno cargado: {PLANETA}   g = {G} m/s²   ρ₀ = {RHO0} kg/m³   γ = {GAMMA}°")

## Paso 2 — La aeronave

Esta es la única celda que editas en todo el archivo. Escribe los datos de **una** aeronave y ejecuta todo lo que sigue.

| Aeronave | Masa estructural [kg] | Área escudo [m²] | Área dosel [m²] | Empuje [kN] | Propelente [kg] |
|---|---|---|---|---|---|
| Clase Atlas | 4 400 | 26,0 | 70 | 120 | 1 600 |
| Clase Terran | 2 600 | 10,0 | 200 | 40 | 900 |
| Clase Huygens | 1 400 | 2,0 | 8 | 4 | 200 |
| Clase Venera | 4 200 | 9,0 | 20 | 60 | 800 |
| Clase Manta | 3 300 | 30,0 | 900 | 18 | 700 |

Las tres constantes de la segunda línea son iguales para toda la flota: el coeficiente de arrastre del escudo, el del dosel y el impulso específico del motor. Los tres límites de la tercera línea son los de la estructura y tampoco dependen de la aeronave.

### Qué hace cada dato

Antes de escribir nada, conviene saber qué toca cada número. Ninguno actúa en una sola fase, y ahí está la dificultad del ejercicio.

| Dato | Dónde actúa | Si es mayor… | Si es menor… |
|---|---|---|---|
| **Masa seca** | las tres fases | frena peor, cae más rápido, el TWR baja y el Δv disponible baja | mejora casi todo, pero es menos nave |
| **Área del escudo** | fase 1 | frena antes y más arriba, menos presión sobre el escudo, el dosel se abre más alto | penetra más profundo antes de frenar |
| **Área del dosel** | fase 2 | entrega más lento a los motores, pero más carga sobre el dosel y descenso más largo | entrega más rápido y la fase 3 gasta más |
| **Empuje** | fase 3 | el TWR sube en proporción directa | menos autoridad para sostenerse |
| **Propelente** | las tres fases | más Δv y más segundos de encendido, pero sube la masa total | menos margen en la fase final |

Tres cosas que conviene tener presentes mientras comparas aeronaves:

**El escudo y la masa se pelean.** Las dos entran en el mismo número, el coeficiente balístico $eta = m/(C_d A)$, una arriba y otra abajo. Ese número resume en una sola cifra lo difícil que es frenar el vehículo, y es el que más manda en la fase 1. Fíjate en el que imprime la celda de abajo.

**El empuje aparece dos veces, y con signos opuestos.** Sube el TWR de la gráfica 6, pero acorta los segundos de encendido que aparecen en el título de la gráfica 7, porque el propelente se agota más rápido. El motor más potente es también el que antes se queda seco.

**El propelente ayuda y estorba a la vez.** Da más Δv y más tiempo de encendido, pero engorda la masa total, que es justo lo que arruina las fases 1 y 2. Y su aporte al Δv es logarítmico: cada kilo extra rinde menos que el anterior.

No hay ninguna combinación que gane en todo. Cada aeronave de la flota es una apuesta distinta sobre qué vale la pena sacrificar.

In [ ]:
NOMBRE      = "Clase Manta"
MASA_SECA   = 3300.
AREA_ESCUDO = 30.
AREA_DOSEL  = 900.
EMPUJE      = 18.
PROPELENTE  = 700.

CD_ESCUDO, CD_DOSEL, ISP = 1.45, .62, 300.
LIM_G, LIM_Q, LIM_DOSEL, LIM_DESC = 15., 25., 289., 3600.
MASA_TOTAL = MASA_SECA + PROPELENTE

print(f"{NOMBRE}   {MASA_TOTAL:.0f} kg   β = {MASA_TOTAL/(CD_ESCUDO*AREA_ESCUDO):.0f} kg/m²")

---
## Fase 1 — Entrada atmosférica

Desde la interfaz de entrada, frenando solo con el escudo térmico. Aquí se disipa casi toda la energía del vehículo.

El paracaídas no se abre a una altura fija: se abre cuando el vehículo baja de **Mach 2**, sea donde sea. Por encima de esa velocidad la tela no sobrevive al inflado. Así que la altura de despliegue no es un dato, es el primer resultado de diseño: la calcula la aeronave.

**Gráfica 1 — Trayectoria.** El diamante marca dónde se abre el dosel. Cuanto más alto, más margen queda para todo lo que viene después. Si la curva llega a la línea roja sin haber bajado de Mach 2, esta aeronave no puede usar paracaídas en este planeta.

**Gráfica 2 — Escudo térmico.** La presión dinámica máxima. Por encima de 25 kPa el material ablativo se desprende. Este número sí depende fuertemente del diseño: lo gobierna el coeficiente balístico β = m / (C_d·A). Un vehículo pesado detrás de un escudo pequeño tiene β alto, penetra más antes de frenar y sufre más presión.

**Gráfica 3 — Carga estructural.** El pico de desaceleración. Por encima de 15 g el vehículo se parte.

Cambia de aeronave y vuelve a mirar la gráfica 3. El número apenas se mueve. Hay una razón, y es la pregunta 1 de la discusión.

In [ ]:
E, V_DESP, H_DESP, _ = volar(H_ENTRADA, V_ENTRADA, 0., MASA_TOTAL, CD_ESCUDO, AREA_ESCUDO, GAMMA, mach=MACH_DOSEL)
CARGA_G, PRESION = E["n"].max(), E["q"].max()/1000
VIABLE = H_DESP > H_SUELTA

f, ax = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
ax[0].plot(E["v"]/1000, E["h"]/1000, c="#2980b9")
ax[0].axhline(H_SUELTA/1000, ls="--", c="#c0392b", label="Altura mínima útil")
if VIABLE: ax[0].scatter(V_DESP/1000, H_DESP/1000, c="gold", marker="D", s=90, ec="k", zorder=5, label=f"Dosel a {H_DESP/1000:.1f} km")
ax[0].set(title="1 · Trayectoria" + (f" — dosel a {H_DESP/1000:.1f} km" if VIABLE else " — NO alcanza Mach 2"),
          xlabel="Velocidad [km/s]", ylabel="Altitud [km]")
ax[0].invert_yaxis(); ax[0].legend(fontsize=7)

ax[1].plot(E["t"], E["q"]/1000, c="#c0392b")
ax[1].axhline(LIM_Q, ls="--", c="k", label=f"Límite {LIM_Q:.0f} kPa")
ax[1].set(title=f"2 · Escudo térmico — pico {PRESION:.0f} kPa", xlabel="Tiempo [s]", ylabel="Presión [kPa]"); ax[1].legend(fontsize=7)

ax[2].plot(E["t"], E["n"], c="#8e44ad")
ax[2].axhline(LIM_G, ls="--", c="k", label=f"Límite {LIM_G:.0f} g")
ax[2].set(title=f"3 · Carga estructural — pico {CARGA_G:.1f} g", xlabel="Tiempo [s]", ylabel="Carga [g]"); ax[2].legend(fontsize=7)
plt.show()

print(f"pico {CARGA_G:.1f} g   presión {PRESION:.0f} kPa   β {MASA_TOTAL/(CD_ESCUDO*AREA_ESCUDO):.0f} kg/m²")
print(f"dosel a {H_DESP/1000:.1f} km y {V_DESP:.0f} m/s" if VIABLE
      else f"llega al suelo a {V_DESP:.0f} m/s, Mach {V_DESP/SONIDO:.1f}: el paracaídas nunca se abre")

---
## Fase 2 — Descenso en paracaídas

Desde donde se abrió el dosel hasta 1,8 km. A los 20 segundos el vehículo suelta el escudo térmico, que ya no sirve y tapa el radar.

**Gráfica 4 — Carga de apertura.** El dosel pasa de plegado a inflado en menos de un segundo y toda la fuerza se descarga de golpe sobre las líneas de suspensión. Es el instante más violento del descenso completo. El límite son 289 kN.

**Gráfica 5 — Descenso bajo dosel.** La línea punteada es la velocidad terminal, donde el arrastre iguala al peso. Si la curva llena se pega a la punteada, el vehículo ya cae tan lento como puede: el paracaídas no dará más. El diamante marca con qué velocidad se entrega a los motores, y el título dice cuánto tardó en bajar.

Los dos parámetros que mandan aquí son el área del dosel y la masa total: el área decide cuánto frena y cuánto tira de las líneas, y la masa decide a qué velocidad se estabiliza la caída.

Aquí hay dos límites opuestos y hay que quedarse entre ellos. Cuanto más rápido llegue a 1,8 km, más combustible tendrá que gastar la fase siguiente. Y cuanto más tarde en llegar, más batería y más exposición térmica consume por el camino: el descenso completo no puede pasar de **60 minutos**, que es lo que dura la energía de a bordo.

En una atmósfera tenue ese segundo límite no se acerca siquiera. En una densa puede ser el que decida la misión entera.

In [ ]:
if VIABLE:
    D, V_FINAL, _, T_DESC = volar(H_DESP, V_DESP, H_SUELTA, MASA_TOTAL, CD_DOSEL, AREA_DOSEL, 90., MASA_SECA*.06, 20.)
    APERTURA, V_TERMINAL = D["f"].max(), D["vt"][-1]
    f, ax = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
    ax[0].plot(D["t"], D["f"], c="#27ae60")
    ax[0].axhline(LIM_DOSEL, ls="--", c="k", label=f"Límite {LIM_DOSEL:.0f} kN")
    ax[0].set(title=f"4 · Carga sobre el dosel — pico {APERTURA:.0f} kN", xlabel="Tiempo [s]", ylabel="Carga [kN]"); ax[0].legend(fontsize=7)
    ax[1].plot(D["v"], D["h"]/1000, c="#8e44ad", label="Velocidad real")
    ax[1].plot(D["vt"], D["h"]/1000, ls=":", c="#7f8c8d", label="Velocidad terminal")
    ax[1].scatter(V_FINAL, H_SUELTA/1000, c="gold", marker="D", s=90, ec="k", zorder=5, label=f"Entrega a {V_FINAL:.0f} m/s")
    ax[1].set(title=f"5 · Descenso bajo dosel — {T_DESC/60:.0f} min de {LIM_DESC/60:.0f}", xlabel="Velocidad [m/s]", ylabel="Altitud [km]"); ax[1].legend(fontsize=7)
    plt.show()
    print(f"carga {APERTURA:.0f} kN   v. terminal {V_TERMINAL:.0f} m/s   entrega a {V_FINAL:.0f} m/s   duración {T_DESC/60:.0f} min de {LIM_DESC/60:.0f}")
else:
    APERTURA = V_TERMINAL = T_DESC = float("nan"); V_FINAL = V_DESP
    print("Sin fase de descenso: esta aeronave llega al suelo por encima de Mach 2.")
    print("La fase 3 se calcula igual, para ver cuánto Δv haría falta para salvar la situación con motores.")

---
## Fase 3 — Aterrizaje propulsado

De 1,8 km al suelo, solo con motores. Se resuelve con dos balances, sin integrar.

**Gráfica 6 — Empuje contra peso.** Si el empuje no supera el peso, el vehículo cae con los motores encendidos. Es una condición de sí o no, y depende de la gravedad local: el mismo motor da un número distinto en cada planeta.

**Gráfica 7 — Presupuesto de velocidad.** Cuánto Δv puede dar el vehículo frente a cuánto necesita:

$$\Delta v_{disp} = I_{sp}\,g_0\,\ln\!\left(\frac{m_{total}}{m_{seca}}\right) \qquad \Delta v_{req} = v_{entrega} + g\,(t_{hover}+15)$$

El primero es la ecuación de Tsiolkovski. Ese logaritmo es la mala noticia de toda la astronáutica: para duplicar el Δv hay que elevar al cuadrado la fracción de masa, así que cargar más tanque da cada vez menos a cambio.

El segundo término del requisito es la pérdida gravitatoria. Mientras los motores frenan también sostienen el peso del vehículo, y ese esfuerzo consume combustible sin producir ninguna desaceleración. En Marte se llevó 161 de los 301 m/s que necesitó Curiosity.

El título de la gráfica trae además los segundos de encendido disponibles frente a los 30 que exige bajar la carga al suelo:

$$t_b = \frac{m_p\,I_{sp}\,g_0}{F}$$

Mira dónde está el empuje en esa fórmula y dónde está en la del TWR. Aparece arriba en una y abajo en la otra: lo que arregla la gráfica 6 lo rompe en la 7. Esa tensión es el problema central de la fase.

In [ ]:
DV_REQ = V_FINAL + G*(HOVER + 15.)
TWR = EMPUJE*1000./(MASA_TOTAL*G)
DV = ISP*9.81*np.log(MASA_TOTAL/MASA_SECA)
T_ENC = PROPELENTE*ISP*9.81/(EMPUJE*1000.)

f, ax = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
ax[0].bar(["Peso", "Empuje"], [MASA_TOTAL*G/1000, EMPUJE], color=["#c0392b", "#27ae60"])
ax[0].set(title=f"6 · Balance vertical — TWR {TWR:.2f}", ylabel="Fuerza [kN]")
ax[0].text(.5, .93, "sostiene" if TWR >= 1 else "no sostiene", transform=ax[0].transAxes, ha="center",
           weight="bold", color="#27ae60" if TWR >= 1 else "#c0392b")
ax[1].bar(["Δv disponible", "Δv requerido"], [DV, DV_REQ], color=["#2980b9", "#e67e22"])
ax[1].set(title=f"7 · Presupuesto de velocidad — encendido {T_ENC:.0f} s de {HOVER:.0f} s", ylabel="Δv [m/s]")
plt.show()

print(f"TWR {TWR:.2f}   Δv {DV:.0f} de {DV_REQ:.0f} m/s   encendido {T_ENC:.0f} s de {HOVER:.0f} s")
print(f"de los {DV_REQ:.0f} m/s requeridos, {G*(HOVER+15.):.0f} son pérdida gravitatoria")

---
## Resumen de esta aeronave

Ejecuta la celda, copia la columna a tu ficha, vuelve al Paso 2 y cambia de aeronave.

In [ ]:
def fila(n, v, l, c):
    vs = "  —  " if v != v else f"{v:9.1f}"
    ls = "   —  " if l is None else ("    —  " if l != l else f"{l:9.1f}")
    cs = " — " if c is None else ("sí " if c else "NO ")
    print(f"{n:24s}{vs}{ls}   {cs}")

print(f"{NOMBRE}  en  {PLANETA}\n")
print(f"{'':24s}{'valor':>9s}{'límite':>9s}   cumple")
fila("Carga de pico [g]",        CARGA_G,    LIM_G,     CARGA_G <= LIM_G)
fila("Presión dinámica [kPa]",   PRESION,    LIM_Q,     PRESION <= LIM_Q)
fila("Altura del dosel [km]",    H_DESP/1000, H_SUELTA/1000, VIABLE)
fila("Carga sobre dosel [kN]",   APERTURA,   LIM_DOSEL, APERTURA <= LIM_DOSEL if VIABLE else False)
fila("Velocidad terminal [m/s]", V_TERMINAL, None,      None)
fila("Descenso [min]",           T_DESC/60 if VIABLE else float("nan"), LIM_DESC/60, T_DESC <= LIM_DESC if VIABLE else False)
fila("Empuje-peso",              TWR,        1.,        TWR >= 1.)
fila("Delta-v [m/s]",            DV,         DV_REQ,    DV >= DV_REQ)
fila("Encendido [s]",            T_ENC,      HOVER,     T_ENC >= HOVER)
print(f"\nMasa total: {MASA_TOTAL:.0f} kg")

---
## Ficha de decisión

Una fila por aeronave.

| Aeronave | Masa [kg] | Carga [g] | Presión [kPa] | Dosel [km] | Carga dosel [kN] | Descenso [min] | TWR | Δv [m/s] | Encendido [s] | Cumple |
|---|---|---|---|---|---|---|---|---|---|---|
| Clase Atlas | 6 000 | | | | | | | | | /8 |
| Clase Terran | 3 500 | | | | | | | | | /8 |
| Clase Huygens | 1 600 | | | | | | | | | /8 |
| Clase Venera | 5 000 | | | | | | | | | /8 |
| Clase Manta | 4 000 | | | | | | | | | /8 |

La columna de masa ya está llena: es la masa total de cada aeronave, estructura más propelente. Es la que decide el empate.

---

## Discusión de equipo

1. **El número que no se mueve.** Cambiaste cinco aeronaves con masas, escudos y coeficientes balísticos muy distintos, y el pico de carga en g salió casi igual en todas. No es un error del modelo: en 1958, Allen y Eggers demostraron que en una entrada balística el pico de desaceleración no depende del vehículo, solo de la velocidad de entrada, del ángulo y de la escala de altura de la atmósfera. Con la gráfica 3 y la tabla del entorno, expliquen qué consecuencia tiene esto para un diseñador: si el pico de g no se puede cambiar rediseñando la nave, ¿qué sí se puede cambiar?

2. **El rediseño.** Elijan la aeronave que peor se comporta en Tierra y arréglenla cambiando **un solo dato** de la ficha. Consigan que cumpla un criterio que antes fallaba, y anoten qué otro criterio empeoró al hacerlo. Ningún cambio sale gratis; el trabajo del ingeniero es elegir qué empeora.

3. **El doble de carga.** Si tuvieran que aterrizar el doble de masa en Tierra, ¿qué se rompe primero: el escudo, el dosel o el tanque? Justifíquenlo con una de las siete gráficas, no con intuición.

---

## Qué se presenta

Tres o cuatro minutos por equipo:

- **El entorno.** Qué planeta les tocó y por qué aterrizar ahí es difícil. Un dato concreto, no una descripción.
- **El método.** Qué evaluaron en cada fase y qué supuestos aceptaron.
- **La decisión.** Qué aeronave recomiendan, qué criterio descartó a cada una de las otras cuatro, y si hubo empate, por cuánta masa lo ganaron. Muestren dos de las siete gráficas: la que sostiene su elección y la que condena a la descartada más cercana.

Escuchen a los otros equipos. Las cinco aeronaves son las mismas para todos y el ganador de cada planeta es distinto. La pregunta del cierre es por qué.

---

Los valores del entorno y de las aeronaves tienen fines de entrenamiento: están construidos para que el ejercicio funcione y no corresponden necesariamente a los de los cuerpos planetarios citados.